# 消融实验分析（模块 11）

读取 `eval/results/ablation_results.jsonl`，产出报告图表：
1. 消融矩阵 A–E 各指标对比表；
2. C 组 vs A 组幻觉率对照（招牌图）；
3. E 组 token 成本对照（压缩收益）；
4. 分角色辩论成本明细。

In [ ]:
import json, pandas as pd
rows = [json.loads(l) for l in open('../eval/results/ablation_results.jsonl', encoding='utf-8')]
df = pd.DataFrame(rows)
df.head()

In [ ]:
import sys; sys.path.insert(0, '..')
from eval.metrics import aggregate, render_markdown
table = aggregate([type('R', (), r)() for r in rows])
print(render_markdown(table))

In [ ]:
# 幻觉率对照（A vs C 是核心对照）
import matplotlib.pyplot as plt
for cfg in ('A','B','C','D','E'):
    sub = df[df.config==cfg]
    if len(sub):
        plt.bar(cfg, sub.numeric_support_rate.mean(), yerr=sub.numeric_support_rate.std())
plt.ylabel('数字溯源命中率'); plt.show()

In [ ]:
# Walk-forward 衰减曲线（数据由量化任务结果导出）
# 读取 backend.quant.walkforward 输出的 folds JSON 绘制训练/样本外夏普并列柱状图